# TransformerLens `HookedTransformer` tutorial

Covers the core mechanics you need for interpretability work:

1. Loading a small model onto MPS (Apple Silicon GPU)
2. A basic forward pass
3. Grabbing activations with `run_with_cache`
4. Intervening on activations with `run_with_hooks` (causal ablation)
5. Gradients: parameter grads via `loss.backward()`, and activation grads via `bwd_hooks`

Model used: **gpt2-small** (124M params, 12 layers) — the standard TransformerLens tutorial model, small enough to run interactively on an M2 Mac. Even tinier options exist if you want faster iteration: `attn-only-1l`, `attn-only-2l`, `solu-1l` (single-layer toy models trained by the TransformerLens authors specifically for interpretability).

In [1]:
import torch
from transformer_lens import HookedTransformer, utils

torch.manual_seed(0)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


/var/folders/h6/l21knyzn0j1919wpph_4f6qr0000gn/T/ipykernel_98708/3667593539.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


**Note on MPS:** TransformerLens will print `UserWarning: MPS backend may produce silently incorrect results` on load — this is a known caveat of PyTorch's MPS backend, not specific to what we're doing here. For learning the API and quick iteration it's fine; for a result you plan to trust/publish, double check it against `device="cpu"` first. You'll also see a couple of harmless deprecation warnings (`utils` → `transformer_lens.utilities`, `from_pretrained` → `TransformerBridge`) — both APIs are still fully supported in this version (3.9.0) and are what this notebook uses.

## 1. Load the model

`HookedTransformer.from_pretrained` downloads the weights from HuggingFace and converts them into TransformerLens's own module structure, where every internal activation (attention pattern, MLP output, residual stream, etc.) is wrapped in a `HookPoint` you can read from or write to.

In [2]:
model = HookedTransformer.from_pretrained("gpt2", device=device)

print(model.cfg.n_layers, "layers,", model.cfg.n_heads, "heads,", model.cfg.d_model, "d_model")

/var/folders/h6/l21knyzn0j1919wpph_4f6qr0000gn/T/ipykernel_98708/819644286.py:1: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)
/Users/sohanthuumala/miniconda3/envs/tlens/lib/python3.11/site-packages/transformer_lens/config/hooked_transformer_config.py:394: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.14.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
12 layers, 12 heads, 768 d_model


## 2. Basic forward pass

`model.to_tokens` handles tokenization (and prepends BOS by default). Calling the model like a function runs a standard forward pass and returns logits of shape `[batch, seq, vocab]`.

In [3]:
prompt = "The quick brown fox jumps over the lazy"
tokens = model.to_tokens(prompt)
print("tokens shape:", tokens.shape)
print("tokens:", model.to_str_tokens(prompt))

logits = model(tokens)
print("logits shape:", logits.shape)

next_token_logits = logits[0, -1]
predicted = model.to_string(next_token_logits.argmax())
print("predicted next token:", repr(predicted))

tokens shape: torch.Size([1, 9])
tokens: ['<|endoftext|>', 'The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy']
logits shape: torch.Size([1, 9, 50257])
predicted next token: ','


## 3. Grabbing activations with `run_with_cache`

`run_with_cache` runs the forward pass and returns `(output, cache)`, where `cache` is an `ActivationCache` — a dict-like object holding every intermediate activation, keyed by hook name (e.g. `blocks.0.hook_resid_post`). `utils.get_act_name(name, layer)` builds these keys for you, and the cache also supports the shorthand `cache[name, layer]`.

In [4]:
logits, cache = model.run_with_cache(tokens)

# Residual stream after layer 0, indexed via utils.get_act_name
resid_post_0 = cache[utils.get_act_name("resid_post", 0)]
print("resid_post layer 0:", resid_post_0.shape)  # [batch, seq, d_model]

# Attention pattern for layer 0, using the (name, layer) shorthand
attn_pattern_0 = cache["pattern", 0]
print("attention pattern layer 0:", attn_pattern_0.shape)  # [batch, n_heads, dest_pos, src_pos]

# MLP output for the last layer
mlp_out_last = cache["mlp_out", model.cfg.n_layers - 1]
print("mlp_out last layer:", mlp_out_last.shape)  # [batch, seq, d_model]

resid_post layer 0: torch.Size([1, 9, 768])
attention pattern layer 0: torch.Size([1, 12, 9, 9])
mlp_out last layer: torch.Size([1, 9, 768])


## 4. Intervening with `run_with_hooks`

A hook is just a function `(activation_tensor, hook) -> activation_tensor` that you attach to a named hook point. `run_with_hooks` attaches it for a single forward pass and removes it afterward. Below we zero-ablate one attention head's output (`hook_z`, shape `[batch, seq, n_heads, d_head]`) and see how much it moves the next-token prediction.

In [5]:
layer_to_ablate = 0
head_to_ablate = 5

def ablate_head_hook(value, hook):
    # value: [batch, seq, n_heads, d_head]
    value[:, :, head_to_ablate, :] = 0.0
    return value

original_logits = model(tokens)
ablated_logits = model.run_with_hooks(
    tokens,
    fwd_hooks=[(utils.get_act_name("z", layer_to_ablate), ablate_head_hook)],
)

orig_probs = original_logits[0, -1].softmax(-1)
ablated_probs = ablated_logits[0, -1].softmax(-1)

print("original top token:", repr(model.to_string(orig_probs.argmax())))
print("ablated top token:", repr(model.to_string(ablated_probs.argmax())))
print("max prob change:", (orig_probs - ablated_probs).abs().max().item())

original top token: ','
ablated top token: ' fox'
max prob change: 0.009911637753248215


## 5. Gradients

### 5a. Parameter gradients

Nothing TransformerLens-specific here — `HookedTransformer` weights are normal `nn.Parameter`s. Run with `return_type="loss"` to get next-token-prediction loss directly, then call `.backward()` as usual.

In [6]:
model.reset_hooks()
model.zero_grad()

loss = model(tokens, return_type="loss")
print("loss:", loss.item())

loss.backward()

w_q_grad = model.blocks[0].attn.W_Q.grad
print("W_Q layer 0 grad shape:", w_q_grad.shape)
print("W_Q layer 0 grad norm:", w_q_grad.norm().item())

loss: 5.309131622314453
W_Q layer 0 grad shape: torch.Size([12, 768, 64])
W_Q layer 0 grad norm: 2.451601266860962


### 5b. Activation gradients (`bwd_hooks`)

To get the gradient flowing into an *intermediate activation* (not a parameter), attach a backward hook via `bwd_hooks=[...]` in `run_with_hooks`. TransformerLens registers this directly on the activation tensor with `tensor.register_hook`, so it survives after `run_with_hooks` returns — call `.backward()` on the result afterward and the hook fires during autograd.

In [7]:
model.reset_hooks()
model.zero_grad()

activation_grads = {}

def save_grad_hook(grad, hook):
    activation_grads[hook.name] = grad.detach().clone()

target_layer = 6
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    bwd_hooks=[(utils.get_act_name("resid_post", target_layer), save_grad_hook)],
    reset_hooks_end=False,  # keep the backward hook attached past this call, until after .backward()
)

loss.backward()
model.reset_hooks()  # now safe to clean up

grad = activation_grads[utils.get_act_name("resid_post", target_layer)]
print("resid_post layer", target_layer, "grad shape:", grad.shape)  # [batch, seq, d_model]
print("grad norm:", grad.norm().item())

resid_post layer 6 grad shape: torch.Size([1, 9, 768])
grad norm: 0.13382893800735474


## 6. Cleanup

Always `reset_hooks()` between experiments if you attached any hooks manually (outside of `run_with_hooks`, which cleans up after itself), and `zero_grad()` before a fresh backward pass.

In [8]:
model.reset_hooks()
model.zero_grad()
print("hooks reset, grads zeroed")

hooks reset, grads zeroed
